# Migrate a daily CSV pipeline to Iceberg with hidden partitioning and prove row-level parity on a 5-day overlap

You inherited the legacy daily aggregation pipeline that has been writing CSVs partitioned by date for three years. Five days ago the team agreed to migrate it to Iceberg with hidden partitioning by ingestion timestamp. The new pipeline has been running alongside the legacy one for the full five-day overlap. Friday morning the legacy gets decommissioned. Tonight you want to be confident.

This lab refuses to let you declare success without the parity diff returning zero rows.

You have four deliverables:
1. Five days of data on both sides. A legacy CSV under `legacy/order_date=YYYY-MM-DD/orders.csv` for each of the five dates, and at least five Iceberg snapshots, one per ingestion day.
2. The same analyst-visible schema on both tables. The Iceberg table's hidden partition column (`_ingestion_date`) exists in the table but the analyst's query does not need to know about it.
3. A parity diff that materializes a `parity/diff/` CTAS via FULL OUTER JOIN on `order_id` with a one-second tolerance on `created_at`. The CTAS returns zero rows.
4. A cutover plan markdown at `parity/cutover_plan.md` with four required H2 sections.

**Pragmatic deviation from the brief.** The brief specifies real Glue ETL jobs running 5x each side. That math lands at roughly 50 minutes of wall clock just for the job runs (2 DPU x ~2 min x 10 runs plus 10 minutes of Glue cold start per run), which would blow the 75-minute lab budget. This notebook replaces the Glue ETL jobs with notebook-local Python helpers that produce the same outputs: a `run_legacy_day(date)` that writes one CSV per partition, and a `run_new_day(date)` that issues `INSERT INTO orders_iceberg SELECT ... FROM seed_table WHERE order_date = ...`. The pedagogical point (writing two side-by-side outputs and diffing them) is preserved; the Glue-ETL-DPU-billed-by-the-minute part is not what the lab is teaching.

Voice from the brief: migration-careful. The parity diff is the gate.

**Time.** Plan for about 75 minutes hands-on. Most of the time goes into reading the parity-diff SQL and writing the cutover plan. The Athena queries each take 3 to 8 seconds. Session window is 90 minutes.

**Cost.** This lab costs about 5 cents on a clean run and lands around 30 to 60 cents with realistic re-runs and debugging. Athena scans on a few hundred rows are well under any meaningful pricing threshold; the floor is the per-query rounded scan minimum. Glue catalog operations are free at this volume. S3 storage on the Iceberg metadata and a dozen tiny CSVs is fractions of a penny. The cleanup cell at the bottom tears down every table, the workgroup, the database, and the bucket so nothing keeps accruing after you finish.

This lab maps to the Data Engineering job-prep track, Pipeline Engineering sub-track. Migration with dual-write and parity proof is the senior-DE move the cert tracks leave out; this lab is the interview talking point that gets you the offer.


In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.8.0


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12. Glue identifiers (database name, table names) use underscores
# because the Athena DDL parser rejects hyphens in identifiers; the bucket
# name and workgroup name use the hyphenated lab slug.

import atexit
import csv
import getpass
import io
import json
import re
import time
from datetime import date, datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "pipeline-migration-csv-to-iceberg"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal the brief's slug field exactly
REGION = "us-east-1"  # all labs in this track run in us-east-1

# Glue database and Glue table identifiers use underscores; Athena DDL parser
# rejects hyphens inside identifiers. See LAB-CREATION-BLUEPRINT.md and the
# Lab 05 athena-cost-optimization sibling for the canonical pattern.
DB = f"labrun_{LAB_ID.replace('-', '_')}_db"
SEED_TABLE = "orders_seed"
LEGACY_TABLE = "orders_legacy"
ICEBERG_TABLE = "orders_iceberg"
PARITY_DIFF_TABLE = "parity_diff"

# Workgroup and bucket use hyphens; both names accept them.
WG = f"labrun-{LAB_ID}-wg"
BUCKET = None  # set after STS resolves the account id

# Five distinct dates for the overlap window. Fixed so re-runs land on the
# same partitions and the checkpoint validators can assert exact prefixes.
START_DATE = date(2026, 4, 1)
DAY_COUNT = 5
DATES = [START_DATE + timedelta(days=i) for i in range(DAY_COUNT)]

# Rows per day. Kept small (legacy CSV plus Iceberg INSERTs scale by this).
ROWS_PER_DAY = 60

# Currencies and customer pool. Small enough to keep the data deterministic
# and the parity diff fast.
CURRENCIES = ["USD", "EUR", "GBP", "JPY"]
CUSTOMER_POOL = [f"cust_{i:03d}" for i in range(1, 41)]


In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"All track-data-engineering labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first Athena call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve the bucket name now that the account ID is in hand. S3 bucket names
# must be globally unique; the per-account suffix keeps the lab deterministic.
BUCKET = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET}")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# The manifest is module-level and in strict reverse-creation order: every
# Glue table tears down first (Iceberg via Athena DROP TABLE so the manifest
# files are released, plain Glue tables via glue.delete_table), then the
# Athena workgroup, then the Glue database, then the bucket. No critical
# (hourly-billed) resources in this lab; Athena bills per-query-scanned only.
#
# Pragmatic deviation from the brief: the brief listed two glue_job entries
# (legacy ETL and new ETL). Those jobs are replaced by notebook-local Python
# helpers (see the next cell), so no Glue jobs are ever created and no
# glue_job manifest entries are needed.
#
# labrun-checks 0.8.0 ships AWS adapters for every resource type in this
# manifest, including iceberg_table, glue_table, athena_workgroup,
# glue_database, and s3_bucket. Iceberg tables drop cleanly via Athena DDL
# (DROP TABLE removes both the Glue catalog entry and the Iceberg manifest
# pointers). The _LabAdapter subclass at the bottom of this notebook
# predates the 0.8.0 iceberg_table adapter and is now functionally
# equivalent to the default; a follow-up can drop it and call run_cleanup
# against the bundled adapter directly. The manifest entries declare the
# resource type so the canonical summary, atexit handler, and orphan scan
# all see them.
#
# CleanupResource has no `location` field per labrun_checks._types; the
# S3 prefix for each Iceberg table is stashed in the `extra` dict so the
# adapter (or a manual operator) can locate the stragglers.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="glue_table",
        id=PARITY_DIFF_TABLE,
        region=REGION,
        parent=DB,
        cli_delete_command=(
            f"aws glue delete-table --database-name {DB} --name {PARITY_DIFF_TABLE}"
        ),
    ),
    CleanupResource(
        type="iceberg_table",
        id=ICEBERG_TABLE,
        region=REGION,
        parent=DB,
        extra={"location": f"s3://{BUCKET}/iceberg/", "prefix": "iceberg/"},
        cli_delete_command=(
            f"aws athena start-query-execution --query-string "
            f"\"DROP TABLE IF EXISTS {DB}.{ICEBERG_TABLE}\" "
            f"--work-group {WG} --query-execution-context Database={DB}"
        ),
    ),
    CleanupResource(
        type="glue_table",
        id=LEGACY_TABLE,
        region=REGION,
        parent=DB,
        cli_delete_command=(
            f"aws glue delete-table --database-name {DB} --name {LEGACY_TABLE}"
        ),
    ),
    CleanupResource(
        type="glue_table",
        id=SEED_TABLE,
        region=REGION,
        parent=DB,
        cli_delete_command=(
            f"aws glue delete-table --database-name {DB} --name {SEED_TABLE}"
        ),
    ),
    CleanupResource(
        type="athena_workgroup",
        id=WG,
        region=REGION,
        cli_delete_command=(
            f"aws athena delete-work-group --work-group {WG} "
            f"--recursive-delete-option"
        ),
    ),
    CleanupResource(
        type="glue_database",
        id=DB,
        region=REGION,
        cli_delete_command=f"aws glue delete-database --name {DB}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell at the bottom is the authoritative entry
    point; this is the safety net for kernel crashes. Errors are swallowed
    because atexit handlers must not raise.
    """
    try:
        if "_LabAdapter" in globals():
            run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())
        else:
            run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for leftover
    state with the lab's tag and surface the cleanup command before creating
    any new resources. This prevents duplicate-bucket and duplicate-database
    scenarios that produce confusing AlreadyExists errors mid-lab.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can collide on the bucket name,")
    print("the Glue database, or the Athena workgroup. Run the cleanup cell")
    print("at the bottom of this notebook first, or remove the resources")
    print("manually with the AWS CLI commands the cleanup cell prints,")
    print("then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


In [ ]:
# NBVAL_SKIP
# Shared infrastructure: S3 bucket, Glue database, Athena workgroup with
# Engine v3 (Iceberg writes and snapshot DDL depend on v3; v2 silently
# rolls over to v2 if you do not pin v3 and produces confusing errors).
# Also defines the run_athena helper every task cell uses, plus a seed-data
# generator that both pipelines read from.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Bucket. us-east-1 rejects LocationConstraint; outside us-east-1 you would
# pass CreateBucketConfiguration={"LocationConstraint": REGION}.
try:
    s3.create_bucket(Bucket=BUCKET)
    s3.put_bucket_tagging(
        Bucket=BUCKET,
        Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
    )
    print(f"Bucket created and tagged: {BUCKET}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("BucketAlreadyOwnedByYou",):
        print(f"Bucket {BUCKET} already owned by this account, reusing.")
    else:
        raise

# Glue database.
try:
    glue.create_database(
        DatabaseInput={
            "Name": DB,
            "Description": f"labrun {LAB_ID} database",
        },
    )
    db_arn = f"arn:aws:glue:{REGION}:{ACCOUNT_ID}:database/{DB}"
    glue.tag_resource(
        ResourceArn=db_arn,
        TagsToAdd={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
    print(f"Glue database created and tagged: {DB}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue database {DB} already exists, reusing.")
    else:
        raise

# Athena workgroup. Engine v3 is pinned explicitly. Iceberg INSERT and the
# $snapshots system table both depend on Engine v3; Engine v2 fails with a
# confusing parse error that does not mention Iceberg.
athena_results_uri = f"s3://{BUCKET}/athena-results/"
try:
    athena.create_work_group(
        Name=WG,
        Configuration={
            "ResultConfiguration": {"OutputLocation": athena_results_uri},
            "EnforceWorkGroupConfiguration": True,
            "PublishCloudWatchMetricsEnabled": False,
            "EngineVersion": {"SelectedEngineVersion": "Athena engine version 3"},
        },
        Description=f"Lab workgroup for {LAB_ID}.",
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Athena workgroup created on Engine v3: {WG}")
except ClientError as e:
    code_str = e.response["Error"]["Code"]
    if code_str == "InvalidRequestException":
        athena.update_work_group(
            WorkGroup=WG,
            ConfigurationUpdates={
                "ResultConfigurationUpdates": {"OutputLocation": athena_results_uri},
                "EnforceWorkGroupConfiguration": True,
                "EngineVersion": {"SelectedEngineVersion": "Athena engine version 3"},
            },
            State="ENABLED",
        )
        print(f"Athena workgroup {WG} already exists, reused on Engine v3.")
    else:
        raise


def run_athena(sql: str, timeout_s: int = 180) -> dict:
    """Submit one query against the lab workgroup and poll until terminal.

    Returns the full QueryExecution dict on SUCCEEDED. Raises RuntimeError
    with the StateChangeReason on FAILED/CANCELLED so the caller does not
    silently treat a failed INSERT as a success.
    """
    start = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DB},
        WorkGroup=WG,
    )
    qid = start["QueryExecutionId"]
    deadline = time.time() + timeout_s
    waits = [
        "Athena is iterating through Iceberg snapshots...",
        "Polling Glue for catalog update...",
        "Iceberg metadata layer is committing the new snapshot...",
        "Parity diff is FULL OUTER JOINing across 5 days. Sit tight...",
    ]
    wait_idx = 0
    while time.time() < deadline:
        resp = athena.get_query_execution(QueryExecutionId=qid)
        state = resp["QueryExecution"]["Status"]["State"]
        if state == "SUCCEEDED":
            return resp["QueryExecution"]
        if state in ("FAILED", "CANCELLED"):
            reason = resp["QueryExecution"]["Status"].get("StateChangeReason", "")
            raise RuntimeError(f"Athena query {state}: {reason}\n  SQL: {sql[:200]}")
        if wait_idx < len(waits):
            print(waits[wait_idx])
            wait_idx += 1
        time.sleep(2)
    raise RuntimeError(f"Athena query {qid} did not finish within {timeout_s} seconds.")


def fetch_scalar(sql: str):
    """Run a single-row, single-column query and return the typed value."""
    execution = run_athena(sql)
    qid = execution["QueryExecutionId"]
    rows = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
    if len(rows) < 2:
        return None
    data = rows[1]["Data"][0]
    return data.get("VarCharValue")


# Deterministic order generator. Both pipelines read from the same in-memory
# pool of canonical rows. The pipelines diverge only in how they write the
# data: the legacy writes CSVs partitioned by order_date, the new pipeline
# INSERTs into an Iceberg table with hidden partitioning by _ingestion_date.
def canonical_orders_for_day(day: date) -> list[dict]:
    """Return the canonical row set for a single order_date.

    Deterministic: the same date always returns the same row list, which is
    what makes the parity diff land at zero on a clean run.
    """
    rows: list[dict] = []
    base_order_id = (day - START_DATE).days * 1_000_000 + 1
    base_ts = datetime(day.year, day.month, day.day, 8, 0, 0, tzinfo=timezone.utc)
    for i in range(ROWS_PER_DAY):
        cust = CUSTOMER_POOL[i % len(CUSTOMER_POOL)]
        currency = CURRENCIES[i % len(CURRENCIES)]
        amount_cents = 1000 + (i * 73) % 5000
        # Stagger created_at by 7-second steps so the timestamps are unique.
        created_at = base_ts + timedelta(seconds=i * 7)
        rows.append({
            "order_id": base_order_id + i,
            "customer_id": cust,
            "amount_cents": amount_cents,
            "currency": currency,
            "created_at": created_at,
            "order_date": day,
        })
    return rows


print()
print("Shared infrastructure ready: bucket, database, workgroup (Engine v3).")


## Task 1: Run both pipelines for the 5-day overlap window

The migration plan calls for the legacy CSV pipeline and the new Iceberg pipeline to run side by side for the five days from 2026-04-01 through 2026-04-05. Today is the morning after day five. You need both sides standing up cleanly before you can diff them.

The legacy pipeline (the CSV side) is a `run_legacy_day(day)` helper that you call once per date. For each call it writes one CSV at `s3://{BUCKET}/legacy/order_date=YYYY-MM-DD/orders.csv` containing the day's orders. The CSV columns are `order_id, customer_id, amount_cents, currency, created_at, order_date`. The `created_at` value is written as epoch-seconds-since-1970 (legacy precision; the new pipeline preserves milliseconds).

The new pipeline (the Iceberg side) is split into three steps for the day-loop helper to call:

1. Create a `orders_seed` Glue external table over the legacy CSVs. This is the source of truth both sides will read from; the legacy CSVs ARE the inputs, the Iceberg side reads from them and writes to Iceberg.
2. Create the `orders_iceberg` Iceberg table with the column shape the analyst reads (`order_id BIGINT, customer_id STRING, amount_cents BIGINT, currency STRING, created_at TIMESTAMP, order_date DATE`) plus one extra column `_ingestion_date TIMESTAMP`. The Iceberg table is partitioned by `day(_ingestion_date)`. This is the hidden-partitioning move; the analyst's query against `orders_iceberg` does not reference `_ingestion_date`, and the partition layout on S3 is managed by Iceberg internally.
3. For each of the five dates, run `INSERT INTO orders_iceberg SELECT ... FROM orders_seed WHERE order_date = ...` so one commit per day lands in the Iceberg snapshot history.

After Task 1 you should see five legacy CSVs on S3, five `orders_iceberg` snapshots in `"orders_iceberg$snapshots"`, and the same row counts on both sides.

The Iceberg DDL shape you want:

```sql
CREATE TABLE orders_iceberg (
  order_id BIGINT, customer_id STRING, amount_cents BIGINT,
  currency STRING, created_at TIMESTAMP, order_date DATE,
  _ingestion_date TIMESTAMP
)
LOCATION 's3://.../iceberg/'
TBLPROPERTIES (
  'table_type' = 'ICEBERG',
  'format' = 'PARQUET',
  'partitioning' = 'day(_ingestion_date)'
)
```

The `_ingestion_date` column is what Iceberg partitions by; the column exists in the table (and shows up in `DESCRIBE`), but the analyst's `SELECT order_id, customer_id, ... FROM orders_iceberg` does not reference it. That is the difference between hidden partitioning and Hive-style explicit partitioning.


In [ ]:
# NBVAL_SKIP
# Task 1: run both pipelines for the 5-day overlap.
# The seed external table + Iceberg DDL are scaffolded for you so the day
# loop has both sides to write through. The student-written pieces are the
# CSV writer (legacy) and the INSERT INTO (Iceberg).

def run_legacy_day(day: date) -> None:
    """Write the legacy CSV for a single order_date.

    Output path: s3://{BUCKET}/legacy/order_date=YYYY-MM-DD/orders.csv
    Columns: order_id, customer_id, amount_cents, currency, created_at, order_date
    created_at is written as epoch-seconds (legacy precision; the new
    pipeline preserves milliseconds, which is the parity-diff gotcha).
    """
    rows = canonical_orders_for_day(day)
    key = f"legacy/order_date={day.isoformat()}/orders.csv"

    # YOUR CODE: Build a CSV body in memory with the header
    # order_id,customer_id,amount_cents,currency,created_at,order_date
    # and one row per `rows` entry. Format created_at as
    # int(row['created_at'].timestamp()) so it lands as epoch seconds.
    # Upload via s3.put_object(Bucket=BUCKET, Key=key, Body=...).
    print(f"Wrote legacy CSV: s3://{BUCKET}/{key} ({len(rows)} rows)")


# Create the orders_seed external table (Glue + S3 over the legacy CSVs).
# The Iceberg side reads from this; the analyst would not normally query
# orders_seed, it is the bridge between the CSV writes and the Iceberg
# INSERTs.
run_athena(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {DB}.{SEED_TABLE} (
  order_id BIGINT,
  customer_id STRING,
  amount_cents BIGINT,
  currency STRING,
  created_at BIGINT,
  order_date_str STRING
)
PARTITIONED BY (order_date DATE)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
LOCATION 's3://{BUCKET}/legacy/'
TBLPROPERTIES (
  'skip.header.line.count' = '1',
  'classification' = 'csv'
)
""")
print(f"Created seed external table {DB}.{SEED_TABLE}")


def run_new_day(day: date) -> None:
    """Insert one day's rows into orders_iceberg from the seed table.

    Each call produces one Iceberg snapshot in the snapshot history table.
    The _ingestion_date column is set to the ingestion wall-clock TIMESTAMP
    so day(_ingestion_date) partitions cluster on actual ingestion day.
    """
    # Add the partition so Athena can read the day's CSV from orders_seed.
    # ALTER TABLE ... ADD PARTITION is the cheapest way to hand-register the
    # partition without running a crawler. IF NOT EXISTS makes the call
    # idempotent across re-runs.
    run_athena(f"""
    ALTER TABLE {DB}.{SEED_TABLE}
    ADD IF NOT EXISTS PARTITION (order_date = DATE '{day.isoformat()}')
    LOCATION 's3://{BUCKET}/legacy/order_date={day.isoformat()}/'
    """)

    # YOUR CODE: Run an INSERT INTO {DB}.{ICEBERG_TABLE} SELECT
    #   order_id, customer_id, amount_cents, currency,
    #   from_unixtime(created_at) AS created_at,
    #   order_date,
    #   current_timestamp AS _ingestion_date
    # FROM {DB}.{SEED_TABLE} WHERE order_date = DATE '{day.isoformat()}'
    # Use the run_athena helper.
    print(f"Inserted day {day.isoformat()} into {DB}.{ICEBERG_TABLE}")


# Create the Iceberg table once. The hidden partitioning lives in
# TBLPROPERTIES; the analyst-facing column list does not include the
# partition column in any SELECT that Task 3's parity diff reads.
# YOUR CODE: Run a CREATE TABLE IF NOT EXISTS for {DB}.{ICEBERG_TABLE} with
# columns:
#   order_id BIGINT, customer_id STRING, amount_cents BIGINT,
#   currency STRING, created_at TIMESTAMP, order_date DATE,
#   _ingestion_date TIMESTAMP
# LOCATION 's3://{BUCKET}/iceberg/' and TBLPROPERTIES (
#   'table_type' = 'ICEBERG', 'format' = 'PARQUET',
#   'partitioning' = 'day(_ingestion_date)'
# ). Use the run_athena helper.
print(f"Created Iceberg table {DB}.{ICEBERG_TABLE}")


# Loop the five days. Each day writes one legacy CSV and one Iceberg INSERT
# (one snapshot per day).
for day in DATES:
    run_legacy_day(day)
    run_new_day(day)

print()
print(f"Both pipelines ran for {DAY_COUNT} days. Legacy CSVs are at")
print(f"  s3://{BUCKET}/legacy/order_date=YYYY-MM-DD/orders.csv")
print(f"and the Iceberg table has {DAY_COUNT} snapshots in {DB}.{ICEBERG_TABLE}.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: 5 legacy partitions exist on S3 AND the Iceberg snapshot
# history shows at least 5 commits. The 5-day overlap is the lab's gate
# for proceeding to the parity-diff work in Task 3.

def checkpoint_1(session):
    try:
        # Phase A: legacy side. Confirm one CSV per expected partition.
        missing_partitions: list[str] = []
        for day in DATES:
            prefix = f"legacy/order_date={day.isoformat()}/"
            resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
            if resp.get("KeyCount", 0) == 0:
                missing_partitions.append(prefix)
        if missing_partitions:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Legacy CSV missing for {len(missing_partitions)} of "
                    f"{DAY_COUNT} partitions: {missing_partitions}. "
                    f"Re-run the day loop in Task 1; run_legacy_day should "
                    f"land one CSV per date prefix."
                ),
            )

        # Phase B: Iceberg side. Confirm at least DAY_COUNT snapshots in the
        # snapshot history system table. The exact count can be >= DAY_COUNT
        # if the student re-ran a day's INSERT (which is fine; the parity
        # diff in Task 3 still has to return zero).
        snap_count_str = fetch_scalar(
            f'SELECT count(*) FROM "{DB}"."{ICEBERG_TABLE}$snapshots"'
        )
        try:
            snap_count = int(snap_count_str) if snap_count_str is not None else 0
        except ValueError:
            return CheckpointResult(
                status="error",
                error_reason=f"Could not parse snapshot count: {snap_count_str!r}",
            )
        if snap_count < DAY_COUNT:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Iceberg snapshot history shows {snap_count} commits, "
                    f"expected at least {DAY_COUNT} (one per ingestion day). "
                    f"Re-run the day loop; each INSERT INTO call produces "
                    f"one snapshot."
                ),
            )

        # Phase C: row counts match on a per-day basis. If the legacy side
        # wrote N rows and the Iceberg side INSERTed M, the parity diff in
        # Task 3 cannot return zero unless N == M for every day.
        legacy_count_str = fetch_scalar(
            f"SELECT count(*) FROM {DB}.{SEED_TABLE}"
        )
        iceberg_count_str = fetch_scalar(
            f"SELECT count(*) FROM {DB}.{ICEBERG_TABLE}"
        )
        try:
            legacy_count = int(legacy_count_str) if legacy_count_str is not None else 0
            iceberg_count = int(iceberg_count_str) if iceberg_count_str is not None else 0
        except ValueError:
            return CheckpointResult(
                status="error",
                error_reason=(
                    f"Could not parse row counts: legacy={legacy_count_str!r} "
                    f"iceberg={iceberg_count_str!r}"
                ),
            )
        expected = DAY_COUNT * ROWS_PER_DAY
        if legacy_count != expected:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Legacy side row count is {legacy_count}, expected "
                    f"{expected} ({DAY_COUNT} days x {ROWS_PER_DAY}). "
                    f"Check that the CSV writer emitted every row in the "
                    f"canonical_orders_for_day(day) list."
                ),
            )
        if iceberg_count != expected:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Iceberg row count is {iceberg_count}, expected "
                    f"{expected}. The INSERT INTO loop may have skipped a "
                    f"day or written duplicates. Inspect orders_iceberg by "
                    f"order_date to find the gap."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks three phases: legacy CSVs present per date prefix, Iceberg snapshot count, and row counts on both sides matching `DAY_COUNT x ROWS_PER_DAY`. Read the failure reason. The most common miss is forgetting to include the header line in the CSV (the seed external table sets `skip.header.line.count = 1`, so a CSV without a header row produces an off-by-one count). The second most common miss is forgetting the hidden-partition column in the Iceberg INSERT.

</details>

<details><summary>Hint 2 (direction)</summary>

Three pieces of code to write.

- The CSV writer in `run_legacy_day` uses Python's `csv` module: `io.StringIO()`, `csv.writer(buf)`, write the header row, then write each row as a tuple. The `created_at` field needs `int(row['created_at'].timestamp())` so it serializes as epoch seconds (this is the legacy-precision part the parity diff has to tolerate). Upload via `s3.put_object(Bucket=BUCKET, Key=key, Body=buf.getvalue().encode('utf-8'))`.
- The Iceberg `CREATE TABLE` lives in the cell once, before the day loop. The column list is the analyst's six columns plus `_ingestion_date TIMESTAMP`; `TBLPROPERTIES` carries `'partitioning' = 'day(_ingestion_date)'` plus the standard Iceberg `table_type` and `format`. The LOCATION is `s3://{BUCKET}/iceberg/`.
- The `INSERT INTO` in `run_new_day` selects all six analyst columns from `orders_seed` for the day's partition, converts the epoch-seconds `created_at` back to a `TIMESTAMP` via `from_unixtime(created_at)`, and appends `current_timestamp AS _ingestion_date`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
def run_legacy_day(day: date) -> None:
    rows = canonical_orders_for_day(day)
    key = f"legacy/order_date={day.isoformat()}/orders.csv"

    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow(["order_id", "customer_id", "amount_cents",
                     "currency", "created_at", "order_date"])
    for r in rows:
        writer.writerow([
            r["order_id"],
            r["customer_id"],
            r["amount_cents"],
            r["currency"],
            int(r["created_at"].timestamp()),
            r["order_date"].isoformat(),
        ])
    s3.put_object(
        Bucket=BUCKET,
        Key=key,
        Body=buf.getvalue().encode("utf-8"),
    )
    print(f"Wrote legacy CSV: s3://{BUCKET}/{key} ({len(rows)} rows)")


def run_new_day(day: date) -> None:
    run_athena(f"""
    ALTER TABLE {DB}.{SEED_TABLE}
    ADD IF NOT EXISTS PARTITION (order_date = DATE '{day.isoformat()}')
    LOCATION 's3://{BUCKET}/legacy/order_date={day.isoformat()}/'
    """)

    run_athena(f"""
    INSERT INTO {DB}.{ICEBERG_TABLE}
    SELECT
      order_id,
      customer_id,
      amount_cents,
      currency,
      from_unixtime(created_at) AS created_at,
      order_date,
      current_timestamp AS _ingestion_date
    FROM {DB}.{SEED_TABLE}
    WHERE order_date = DATE '{day.isoformat()}'
    """)
    print(f"Inserted day {day.isoformat()} into {DB}.{ICEBERG_TABLE}")


run_athena(f"""
CREATE TABLE IF NOT EXISTS {DB}.{ICEBERG_TABLE} (
  order_id BIGINT,
  customer_id STRING,
  amount_cents BIGINT,
  currency STRING,
  created_at TIMESTAMP,
  order_date DATE,
  _ingestion_date TIMESTAMP
)
LOCATION 's3://{BUCKET}/iceberg/'
TBLPROPERTIES (
  'table_type' = 'ICEBERG',
  'format' = 'PARQUET',
  'partitioning' = 'day(_ingestion_date)'
)
""")
print(f"Created Iceberg table {DB}.{ICEBERG_TABLE}")
```

If the `INSERT INTO` fails with a partition-not-found error, the `ALTER TABLE ... ADD PARTITION` for that day's `orders_seed` partition did not run; the day loop calls it before the INSERT, so re-run the whole loop. If the CREATE fails with `EXTERNAL_LOCATION_NOT_EMPTY`, an Iceberg table previously lived at `s3://{BUCKET}/iceberg/`; run the cleanup cell at the bottom of the notebook to empty the prefix, then re-run setup.

</details>


**Wallet check.** About two cents so far. Five legacy CSV writes (S3 PutObject), one CREATE for the seed external table, one CREATE for the Iceberg table, five ALTER TABLE ADD PARTITIONs, and five INSERT INTOs. The CSVs are 60 rows each so the scan totals are pennies of pennies. Your morning coffee cost a hundred times more.


## Task 2: Confirm the analyst-visible schema matches on both sides

The analytics team queries the legacy table today with this column projection:

```sql
SELECT order_id, customer_id, amount_cents, currency, created_at, order_date
FROM orders_legacy
WHERE order_date BETWEEN ... AND ...
```

The migration succeeds only if the same `SELECT` against `orders_iceberg` returns the same columns with the same types. The Iceberg table has one extra column `_ingestion_date` for hidden partitioning, but the analyst's query never references it.

This task is the moment to register the legacy external table in Glue (the seed table from Task 1 was a private bridge; the analyst-facing `orders_legacy` table is what they actually query). Both tables must expose: `order_id BIGINT, customer_id STRING, amount_cents BIGINT, currency STRING, created_at TIMESTAMP, order_date DATE`.

There is a gotcha: the legacy CSV stores `created_at` as epoch-seconds and `order_date` as a partition value. The analyst-facing `orders_legacy` table needs to expose `created_at` as a `TIMESTAMP` and `order_date` as a `DATE`. The cleanest move is to register `orders_legacy` as a view over `orders_seed` with the type conversions applied.

After this task: `glue.get_table` returns the six analyst columns on both `orders_legacy` and `orders_iceberg`. The Iceberg table also carries `_ingestion_date` in its column list (Athena Iceberg surfaces partition columns in `DESCRIBE`); the lesson is that hidden partitioning means the analyst's queries do not need to know about partition columns, not that the column is invisible to introspection.


In [ ]:
# NBVAL_SKIP
# Task 2: stand up the analyst-facing orders_legacy view. orders_iceberg is
# already correctly shaped from Task 1. The orders_legacy view applies the
# epoch-seconds-to-TIMESTAMP and string-to-DATE conversions on top of the
# seed external table so the analyst's projection works without changes.

# YOUR CODE: Run a CREATE OR REPLACE VIEW for {DB}.{LEGACY_TABLE} that
# SELECTs from {DB}.{SEED_TABLE} with these expressions:
#   order_id, customer_id, amount_cents, currency,
#   from_unixtime(created_at) AS created_at,
#   order_date
# The view exposes the six analyst columns with the right types; the seed
# table's underlying BIGINT created_at and DATE-partition order_date stay
# in place. Use the run_athena helper.

print(f"Created analyst-facing view {DB}.{LEGACY_TABLE}")

# Quick confirmation print: spot-check the row count of the view matches
# what Task 1 wrote.
view_count = fetch_scalar(f"SELECT count(*) FROM {DB}.{LEGACY_TABLE}")
print(f"View row count: {view_count}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: glue.get_table on both orders_legacy and orders_iceberg.
# Assert the six analyst columns are present in both with the right types.
# The Iceberg table can carry the extra _ingestion_date column; the legacy
# view exposes only the six.

def checkpoint_2(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        required = {
            "order_id": "bigint",
            "customer_id": "string",
            "amount_cents": "bigint",
            "currency": "string",
            "created_at": "timestamp",
            "order_date": "date",
        }

        # orders_legacy
        try:
            legacy_meta = glue_client.get_table(DatabaseName=DB, Name=LEGACY_TABLE)
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Glue table {DB}.{LEGACY_TABLE} does not exist. "
                        f"Task 2 expects a CREATE VIEW that wraps "
                        f"{DB}.{SEED_TABLE} with the type conversions."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        legacy_cols = {
            c["Name"]: c["Type"]
            for c in legacy_meta["Table"]["StorageDescriptor"].get("Columns", [])
        }
        # Views in Glue store the column list under StorageDescriptor.Columns
        # the same way tables do, so the assertion is uniform.
        missing_legacy = [
            f"{name} {required_type}"
            for name, required_type in required.items()
            if name not in legacy_cols
        ]
        if missing_legacy:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"orders_legacy is missing required analyst columns: "
                    f"{missing_legacy}. Found: {sorted(legacy_cols.keys())}."
                ),
            )

        # Type alignment for orders_legacy. Glue stores types lowercased.
        for col, expected_type in required.items():
            actual = legacy_cols[col]
            if actual.lower() != expected_type:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"orders_legacy column {col!r} has type {actual!r}, "
                        f"expected {expected_type!r}. The legacy view needs "
                        f"from_unixtime(created_at) to land created_at as "
                        f"TIMESTAMP and order_date as DATE."
                    ),
                )

        # orders_iceberg
        try:
            iceberg_meta = glue_client.get_table(DatabaseName=DB, Name=ICEBERG_TABLE)
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Glue table {DB}.{ICEBERG_TABLE} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        iceberg_cols = {
            c["Name"]: c["Type"]
            for c in iceberg_meta["Table"]["StorageDescriptor"].get("Columns", [])
        }
        missing_iceberg = [
            f"{name} {required_type}"
            for name, required_type in required.items()
            if name not in iceberg_cols
        ]
        if missing_iceberg:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"orders_iceberg is missing required analyst columns: "
                    f"{missing_iceberg}. Found: {sorted(iceberg_cols.keys())}."
                ),
            )

        # Type alignment for orders_iceberg.
        for col, expected_type in required.items():
            actual = iceberg_cols[col]
            if actual.lower() != expected_type:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"orders_iceberg column {col!r} has type {actual!r}, "
                        f"expected {expected_type!r}. The Iceberg CREATE in "
                        f"Task 1 set the column types; if you re-ran a "
                        f"different DDL, run cleanup and rebuild."
                    ),
                )

        # The Iceberg side should also carry _ingestion_date in its column
        # list; that is the lab's teaching point on hidden partitioning. If
        # the column is missing, the partitioning predicate cannot fire.
        if "_ingestion_date" not in iceberg_cols:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"orders_iceberg is missing the _ingestion_date column. "
                    f"Hidden partitioning needs the column in the table "
                    f"definition; the analyst's query just does not "
                    f"reference it. Re-run the Iceberg CREATE in Task 1."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

Hidden partitioning means the partition column lives in the table definition but the analyst-facing query does not reference it. Read the checkpoint failure reason. If it says `orders_legacy does not exist`, you have not created the view yet. If it says `_ingestion_date is missing`, the Iceberg CREATE in Task 1 was wrong; re-run it. If it says `created_at has type bigint`, the legacy view forgot the `from_unixtime` conversion.

</details>

<details><summary>Hint 2 (direction)</summary>

The orders_legacy table is best registered as a view on top of orders_seed (not a separate physical CSV table) because the seed already holds the data; the view just applies the type conversions the analyst expects. Use `CREATE OR REPLACE VIEW` so re-running the cell is idempotent. The view's SELECT list is exactly the six analyst columns with `from_unixtime(created_at) AS created_at` to convert epoch seconds to a TIMESTAMP. The `order_date` column from the seed table's partition definition is already a DATE; pass it through as-is.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
run_athena(f"""
CREATE OR REPLACE VIEW {DB}.{LEGACY_TABLE} AS
SELECT
  order_id,
  customer_id,
  amount_cents,
  currency,
  from_unixtime(created_at) AS created_at,
  order_date
FROM {DB}.{SEED_TABLE}
""")
print(f"Created analyst-facing view {DB}.{LEGACY_TABLE}")

view_count = fetch_scalar(f"SELECT count(*) FROM {DB}.{LEGACY_TABLE}")
print(f"View row count: {view_count}")
```

If the CREATE fails with `Table already exists`, the previous run created it as a TABLE (not a VIEW). DROP it and re-run: `DROP TABLE IF EXISTS {DB}.{LEGACY_TABLE}`. If the view row count is zero, the seed table's partitions were not registered; re-run Task 1's day loop because `ALTER TABLE ... ADD PARTITION` is what makes the seed table queryable.

</details>


**Wallet check.** About three cents so far. The view creation is one Athena query; the checkpoint runs two `glue.get_table` calls (free) and one scalar `count(*)` query. Glue catalog operations are always free at this volume. Still rounding to zero on any honest pricing calculator.


## Task 3: Materialize the parity diff and prove it returns zero rows

The migration succeeds only if every row in the legacy 5-day window has a matching row in the Iceberg 5-day window, and vice versa. A FULL OUTER JOIN on `order_id` catches both directions: missing-from-Iceberg rows surface as NULL on the Iceberg side, missing-from-legacy rows surface as NULL on the legacy side.

The gotcha is `created_at` precision. The legacy CSV serializes `created_at` as epoch-seconds; the Iceberg side preserves whatever precision was in the source. When you cast both back to a TIMESTAMP for the join, the legacy side is second-aligned and the Iceberg side may include milliseconds. A naive equality predicate (`a.created_at = b.created_at`) returns mismatches on every row. The fix is a one-second tolerance:

```sql
ABS(date_diff('second', a.created_at, b.created_at)) <= 1
```

Run the diff as a `CREATE TABLE AS SELECT` (CTAS) to `parity/diff/` so the diff is persisted (the analytics team will want to inspect it). Then a `SELECT count(*) FROM parity_diff` returns zero, and the checkpoint passes.

The diff query shape:

```sql
CREATE TABLE parity_diff
WITH (
  format = 'PARQUET',
  external_location = 's3://.../parity/diff/'
)
AS
SELECT
  COALESCE(a.order_id, b.order_id) AS order_id,
  a.amount_cents AS legacy_amount,
  b.amount_cents AS iceberg_amount,
  a.created_at AS legacy_created_at,
  b.created_at AS iceberg_created_at,
  CASE
    WHEN a.order_id IS NULL THEN 'missing_from_legacy'
    WHEN b.order_id IS NULL THEN 'missing_from_iceberg'
    WHEN a.amount_cents <> b.amount_cents THEN 'amount_mismatch'
    WHEN ABS(date_diff('second', a.created_at, b.created_at)) > 1 THEN 'created_at_drift'
    ELSE 'unknown'
  END AS reason
FROM orders_legacy a
FULL OUTER JOIN orders_iceberg b ON a.order_id = b.order_id
WHERE a.order_id IS NULL
   OR b.order_id IS NULL
   OR a.amount_cents <> b.amount_cents
   OR ABS(date_diff('second', a.created_at, b.created_at)) > 1
```

The WHERE clause filters the join to only the mismatching rows. On a clean run the result is empty, and `parity_diff` is a zero-row table.


In [ ]:
# NBVAL_SKIP
# Task 3: write the parity diff as a CTAS and confirm zero rows.
# The CTAS materializes the diff at s3://{BUCKET}/parity/diff/ so the
# analytics team can inspect it after the fact, and the checkpoint runs a
# single count(*) against it.

# Drop any prior parity_diff from a previous attempt (CTAS does not support
# CREATE OR REPLACE in Athena's CTAS dialect). The DROP is idempotent.
run_athena(f"DROP TABLE IF EXISTS {DB}.{PARITY_DIFF_TABLE}")

# YOUR CODE: Run a CREATE TABLE {DB}.{PARITY_DIFF_TABLE} AS SELECT ... that
# FULL OUTER JOINs {DB}.{LEGACY_TABLE} (alias a) with {DB}.{ICEBERG_TABLE}
# (alias b) on a.order_id = b.order_id. Project COALESCE(a.order_id,
# b.order_id) AS order_id, a.amount_cents AS legacy_amount, b.amount_cents
# AS iceberg_amount, a.created_at AS legacy_created_at, b.created_at AS
# iceberg_created_at, and a CASE expression for the reason as in the
# markdown above. Filter via WHERE so only mismatching rows survive. Use:
#   WITH (format = 'PARQUET',
#         external_location = 's3://{BUCKET}/parity/diff/')
# as the CTAS properties so the parquet output lands at the right S3 prefix.
# Use the run_athena helper.

print(f"Materialized parity diff at s3://{BUCKET}/parity/diff/")

# Quick spot-check before the checkpoint. The checkpoint runs its own
# count, but printing here helps debugging.
diff_count = fetch_scalar(f"SELECT count(*) FROM {DB}.{PARITY_DIFF_TABLE}")
print(f"Parity diff rows: {diff_count}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: the parity diff table exists and contains zero rows. Both
# sides of the 5-day overlap must agree on every order_id, with a one-second
# tolerance on created_at.

def checkpoint_3(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            glue_client.get_table(DatabaseName=DB, Name=PARITY_DIFF_TABLE)
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Glue table {DB}.{PARITY_DIFF_TABLE} does not exist. "
                        f"Task 3 expects a CTAS that materializes the diff "
                        f"so the analytics team can inspect mismatches."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        diff_count_str = fetch_scalar(
            f"SELECT count(*) FROM {DB}.{PARITY_DIFF_TABLE}"
        )
        try:
            diff_count = int(diff_count_str) if diff_count_str is not None else 0
        except ValueError:
            return CheckpointResult(
                status="error",
                error_reason=f"Could not parse parity_diff count: {diff_count_str!r}",
            )

        if diff_count != 0:
            # Pull a sample of the reason values so the failure message is
            # diagnostic, not just numeric.
            reason_sql = (
                f"SELECT reason, count(*) FROM {DB}.{PARITY_DIFF_TABLE} "
                f"GROUP BY reason ORDER BY count(*) DESC LIMIT 5"
            )
            try:
                exec_dict = run_athena(reason_sql)
                qid = exec_dict["QueryExecutionId"]
                rows = athena.get_query_results(
                    QueryExecutionId=qid
                )["ResultSet"]["Rows"]
                # First row is the header.
                breakdown_pairs = []
                for r in rows[1:]:
                    fields = r.get("Data", [])
                    if len(fields) >= 2:
                        reason = fields[0].get("VarCharValue", "")
                        n = fields[1].get("VarCharValue", "")
                        breakdown_pairs.append(f"{reason}={n}")
                breakdown = ", ".join(breakdown_pairs) if breakdown_pairs else "n/a"
            except Exception:
                breakdown = "(could not enumerate)"

            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Parity diff returned {diff_count} mismatching rows. "
                    f"Breakdown by reason: {breakdown}. The migration is "
                    f"not ready for cutover until this is zero. Inspect "
                    f"the parity_diff table directly for the row-level "
                    f"specifics."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

FULL OUTER JOIN catches rows missing from either side (a.order_id IS NULL means missing from legacy, b.order_id IS NULL means missing from iceberg). The row-by-row tolerance on `created_at` is the gotcha; without it the diff returns one mismatch row per order because the legacy side serialized epoch-seconds and the Iceberg side keeps the converted timestamp at second precision but the join projects them differently.

</details>

<details><summary>Hint 2 (direction)</summary>

The tolerance predicate that absorbs the legacy precision drift is `ABS(date_diff('second', a.created_at, b.created_at)) <= 1`. Use it in two places: the WHERE clause (so rows within tolerance are not flagged as mismatches) and the CASE expression's `created_at_drift` branch (so the diagnostic reason for any actual drift is accurate). The CTAS properties live in a `WITH (...)` block before the `AS`; `format = 'PARQUET'` and `external_location = 's3://{BUCKET}/parity/diff/'` are the two you need.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3:

```python
run_athena(f"DROP TABLE IF EXISTS {DB}.{PARITY_DIFF_TABLE}")

run_athena(f"""
CREATE TABLE {DB}.{PARITY_DIFF_TABLE}
WITH (
  format = 'PARQUET',
  external_location = 's3://{BUCKET}/parity/diff/'
)
AS
SELECT
  COALESCE(a.order_id, b.order_id) AS order_id,
  a.amount_cents AS legacy_amount,
  b.amount_cents AS iceberg_amount,
  a.created_at AS legacy_created_at,
  b.created_at AS iceberg_created_at,
  CASE
    WHEN a.order_id IS NULL THEN 'missing_from_legacy'
    WHEN b.order_id IS NULL THEN 'missing_from_iceberg'
    WHEN a.amount_cents <> b.amount_cents THEN 'amount_mismatch'
    WHEN ABS(date_diff('second', a.created_at, b.created_at)) > 1 THEN 'created_at_drift'
    ELSE 'unknown'
  END AS reason
FROM {DB}.{LEGACY_TABLE} a
FULL OUTER JOIN {DB}.{ICEBERG_TABLE} b ON a.order_id = b.order_id
WHERE a.order_id IS NULL
   OR b.order_id IS NULL
   OR a.amount_cents <> b.amount_cents
   OR ABS(date_diff('second', a.created_at, b.created_at)) > 1
""")
print(f"Materialized parity diff at s3://{BUCKET}/parity/diff/")

diff_count = fetch_scalar(f"SELECT count(*) FROM {DB}.{PARITY_DIFF_TABLE}")
print(f"Parity diff rows: {diff_count}")
```

If the CTAS fails with `external_location must be empty`, a previous attempt left parquet files at `s3://{BUCKET}/parity/diff/`. Empty the prefix manually: `aws s3 rm s3://{BUCKET}/parity/diff/ --recursive`. If the diff returns more than zero, the reason breakdown in the checkpoint failure message tells you which side broke: `created_at_drift` means the tolerance is wrong, `amount_mismatch` means the seed table data is corrupt (probably from a partial Task 1 re-run; rebuild the table).

</details>


**Wallet check.** About four cents so far. The CTAS is one Athena query (the largest of the lab; it FULL OUTER JOINs 300 rows against 300 rows). The checkpoint runs one count plus one optional grouped reason query if the count is nonzero. Total Athena scans for the entire lab so far are under 1 MB. Your morning coffee still costs more.


## Task 4: Write the cutover plan markdown

The parity diff is zero. That is necessary but not sufficient. Before Friday's decommission, the team needs a written cutover plan covering four areas:

1. **Risk inventory.** What can go wrong between now and the legacy shutdown? Be specific: cite the parity diff result, name the downstream consumers that read from the legacy table, name the dashboard owners.
2. **Rollback plan.** If the analyst team flags a problem on Saturday, how do you revert? What artifacts do you keep around to make the revert possible (the legacy S3 prefix, the orders_legacy view, the seed external table)? What is the SLA for the decision to roll back?
3. **Decommission steps.** Exact sequence of cuts: stop the legacy CSV writer, archive the CSV prefix (or delete it), drop the orders_legacy view, deprecate any dashboards that point at the legacy table. Each step gets a checkbox.
4. **Success criteria.** What metric or event tells the team the cutover is done? Three days of zero parity-diff rows? Zero queries against orders_legacy in CloudWatch metrics? A signoff from the analytics lead?

Each of the four sections is an H2 (`## Risk inventory`, etc.). The total file size must be over 1 KB so the checkpoint can confirm a real plan was written, not a placeholder.

Upload the result to `s3://{BUCKET}/parity/cutover_plan.md`.

The hint chain provides a template you can copy as a starting point. Customize the specifics; the checkpoint validates structure (four H2 sections, > 1 KB) rather than wording.


In [ ]:
# NBVAL_SKIP
# Task 4: write the cutover plan markdown to S3.
# The checkpoint asserts the file exists, is over 1 KB, and contains the
# four required H2 sections. The content is yours; the template in Hint 3
# is a starting point.

cutover_plan_key = "parity/cutover_plan.md"

# YOUR CODE: Build a markdown document as a Python string, then upload it
# via s3.put_object(Bucket=BUCKET, Key=cutover_plan_key, Body=...). The
# markdown must contain four H2 sections (lines starting with '## '):
#   ## Risk inventory
#   ## Rollback plan
#   ## Decommission steps
#   ## Success criteria
# Each section needs at least a few sentences of content; the file must be
# over 1 KB total. Hint 3 has a complete template you can copy and adapt.

print(f"Uploaded cutover plan to s3://{BUCKET}/{cutover_plan_key}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: cutover_plan.md exists at s3://{BUCKET}/parity/cutover_plan.md,
# is over 1 KB, and contains the four required H2 section headers.

def checkpoint_4(session):
    try:
        key = "parity/cutover_plan.md"
        try:
            head = s3.head_object(Bucket=BUCKET, Key=key)
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str in ("NoSuchKey", "404", "NotFound"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"s3://{BUCKET}/{key} does not exist. Task 4 expects "
                        f"a markdown file uploaded via s3.put_object."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        size = head.get("ContentLength", 0)
        if size <= 1024:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cutover plan is {size} bytes; checkpoint requires > 1 KB "
                    f"(1024 bytes). A placeholder file is not a plan; write "
                    f"out the four sections with substantive content."
                ),
            )

        body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read().decode("utf-8")

        required_sections = [
            "Risk inventory",
            "Rollback plan",
            "Decommission steps",
            "Success criteria",
        ]
        missing: list[str] = []
        for section in required_sections:
            # Accept the section as an H2 header. The check tolerates extra
            # whitespace but requires the '## ' prefix on its own line.
            pattern = re.compile(rf"^##\s+{re.escape(section)}\s*$", re.MULTILINE)
            if not pattern.search(body):
                missing.append(section)
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cutover plan is missing required H2 sections: {missing}. "
                    f"Each section must appear as a Markdown H2 header on its "
                    f"own line: ## Risk inventory, ## Rollback plan, "
                    f"## Decommission steps, ## Success criteria."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks three properties: file exists at the right S3 key, file is over 1 KB, and four exact H2 section headers are present. Read the failure reason. The most common miss is forgetting the leading `## ` (single hash counts as H1, three hashes count as H3; the validator checks for H2 specifically). The second most common miss is writing a placeholder and not hitting the 1 KB floor.

</details>

<details><summary>Hint 2 (direction)</summary>

Build the markdown as a Python triple-quoted f-string with `## Risk inventory`, `## Rollback plan`, `## Decommission steps`, `## Success criteria` on their own lines. Fill each section with three to five sentences referencing the actual pipeline (the parity diff, orders_legacy, orders_iceberg, the five-day overlap, the seed table). The 1 KB target is easy to hit: a single full paragraph per section gets you there. Upload via `s3.put_object(Bucket=BUCKET, Key=cutover_plan_key, Body=plan.encode('utf-8'))`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
plan = f"""# Cutover plan: orders_legacy -> orders_iceberg

The 5-day parity diff returned zero rows on {date.today().isoformat()}. The migration is ready for the legacy decommission. This document is the operating plan for the cutover.

## Risk inventory

The biggest risk is a downstream dashboard that pins its query to orders_legacy without our knowledge. The CSV writer has run for three years; the analytics team has probably built derivative views that we have not catalogued. Before the decommission we need a Glue Lake Formation query history audit on every table that joins to orders_legacy. The second risk is timestamp precision drift: the parity diff used a one-second tolerance, which is correct for the analytics use case but will produce small ordering differences if a downstream window function depends on strict equality of created_at. Risk three is the seed external table; if it is dropped before orders_iceberg backfills any missing days, we lose the raw CSV input. Keep the seed table for at least 30 days post-cutover.

## Rollback plan

If the analytics lead flags a problem on Saturday or Sunday, the rollback path is: re-enable the legacy CSV writer (it was paused, not deleted), re-CREATE the orders_legacy view (we keep the DDL in the runbook), and point any consumer queries back at orders_legacy. The orders_iceberg table stays in place during the rollback; it is the source of truth going forward, but the legacy view lights up again so consumers do not break. The rollback decision SLA is four hours from the first analyst report. After 72 hours of zero rollback signal, the legacy S3 prefix is archived to Glacier and the rollback path becomes a 12-hour restore rather than an instant flip.

## Decommission steps

- [ ] Stop the legacy daily CSV writer (the Lambda that emits to legacy/order_date=...).
- [ ] Pause for 24 hours. Watch CloudWatch query metrics on orders_legacy; if a query fires from any consumer, halt and investigate.
- [ ] Notify the analytics, BI, and platform teams that orders_legacy is being deprecated.
- [ ] DROP VIEW orders_legacy.
- [ ] Archive the legacy/ S3 prefix to Glacier Deep Archive (do not delete; rollback insurance).
- [ ] Update the data catalog page for orders to remove the legacy entry and reference only orders_iceberg.
- [ ] Send the final cutover-complete announcement.

## Success criteria

The cutover is done when three consecutive nightly parity diffs return zero rows on the most recent ingestion day (one nightly comparison run against the seed table for three nights post-cutover), AND zero queries against orders_legacy fire across that 72-hour window in CloudWatch query metrics, AND the analytics lead signs off in writing. Until all three land, the legacy view stays available in a paused state. The orders_iceberg snapshot history is the operational truth from that signoff forward.
"""

s3.put_object(
    Bucket=BUCKET,
    Key=cutover_plan_key,
    Body=plan.encode("utf-8"),
    ContentType="text/markdown",
)
print(f"Uploaded cutover plan to s3://{BUCKET}/{cutover_plan_key}")
```

If the upload succeeds but the checkpoint fails on the H2 check, copy the markdown from this hint verbatim; the checkpoint requires `## ` (two hashes plus space) on its own line for each section title.

</details>


**Wallet check.** About five cents in total scan-wise. Four checkpoints, around twelve Athena queries in the task cells, plus the checkpoint validators each run two to three scalar queries (plus one grouped breakdown query on a parity-diff failure). Cleanup adds a handful more. The lab will land in the 30 to 60 cent range with realistic re-runs and debugging.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order: Glue tables and the Iceberg table first (Iceberg via Athena
# DROP TABLE so the manifest pointers are released), then the workgroup,
# then the database, then the bucket. No critical (hourly-billed) resources
# in this lab.
#
# labrun-checks 0.8.0 now ships an iceberg_table adapter, so the _LabAdapter
# subclass below is functionally equivalent to the default AwsCleanupAdapter
# and is kept only for parity with this lab's prior version; a follow-up
# can drop it and call run_cleanup against the default adapter directly.

import sys
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter:
    """AwsCleanupAdapter wrapper that adds iceberg_table support.

    Iceberg tables are dropped via Athena DDL because that removes both the
    Glue catalog entry AND the Iceberg manifest + data files in one call.
    A direct glue.delete_table would leave the Iceberg metadata orphaned in
    S3, and the next CREATE TABLE at the same LOCATION would fail with
    EXTERNAL_LOCATION_NOT_EMPTY.

    After the DROP, any straggler objects under the table's S3 prefix get
    swept by the s3_bucket adapter when the bucket itself is deleted.
    """

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def _athena_client(self, credentials: dict):
        return boto3.client(
            "athena",
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def _glue_client(self, credentials: dict):
        return boto3.client(
            "glue",
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def _drop_iceberg(self, credentials: dict, resource) -> None:
        """Issue DROP TABLE IF EXISTS via Athena and poll until terminal."""
        athena_c = self._athena_client(credentials)
        parent = resource.parent or DB
        sql = f"DROP TABLE IF EXISTS {parent}.{resource.id}"
        try:
            start = athena_c.start_query_execution(
                QueryString=sql,
                QueryExecutionContext={"Database": parent},
                WorkGroup=WG,
            )
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str in ("InvalidRequestException", "ResourceNotFoundException"):
                # Workgroup is already gone (cleanup re-run); fall back to a
                # direct glue.delete_table so the catalog entry still goes.
                try:
                    self._glue_client(credentials).delete_table(
                        DatabaseName=parent, Name=resource.id
                    )
                except ClientError as inner:
                    if inner.response["Error"]["Code"] == "EntityNotFoundException":
                        return
                    raise
                return
            raise
        qid = start["QueryExecutionId"]
        deadline = time.time() + 60
        while time.time() < deadline:
            resp = athena_c.get_query_execution(QueryExecutionId=qid)
            state = resp["QueryExecution"]["Status"]["State"]
            if state == "SUCCEEDED":
                return
            if state in ("FAILED", "CANCELLED"):
                reason = resp["QueryExecution"]["Status"].get("StateChangeReason", "")
                if "does not exist" in reason.lower() or "not found" in reason.lower():
                    return
                raise RuntimeError(
                    f"DROP TABLE {parent}.{resource.id} {state}: {reason}"
                )
            time.sleep(1)
        raise RuntimeError(
            f"DROP TABLE {parent}.{resource.id} did not finish within 60 seconds."
        )

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "iceberg_table":
            self._drop_iceberg(credentials, resource)
            return
        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "iceberg_table":
            glue_c = self._glue_client(credentials)
            parent = resource.parent or DB
            try:
                glue_c.get_table(DatabaseName=parent, Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise
        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# Empty the S3 bucket before run_cleanup tears it down. Iceberg leaves
# metadata.json files, snapshot files, and data files; the CTAS parity diff
# wrote parquet to parity/diff/; Athena writes results under athena-results/.
# The bucket adapter cannot delete a non-empty bucket; emptying it here is
# what makes the s3_bucket delete succeed on the first pass.
print("Emptying the S3 bucket before teardown...")
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: approximately $0.05 to $0.40 based on the number of Athena queries actually executed.** Athena is the only metered service in this lab; Glue catalog operations and S3 storage on five tiny CSVs plus Iceberg metadata are effectively free at this volume. The bucket, database, workgroup, view, seed external table, parity_diff table, and Iceberg table were destroyed by the cleanup cell above, so nothing is still accruing. Check your AWS Billing console in 24 hours to confirm the exact number.


## These are not graded. They are for you.

Three questions worth sitting with before your next migration design review.

1. The parity diff used a one-second tolerance on `created_at` to absorb the legacy CSV's epoch-second precision. That tolerance is correct for an analytics use case where timestamps are used for windowing and ordering. Walk through one scenario where the same one-second tolerance would be wrong, and how you would explain that to a stakeholder who reads the parity diff as "zero mismatches = identical data." What would you change about the cutover plan to surface that risk?

2. Hidden partitioning on Iceberg means the analyst's query does not reference `_ingestion_date`, but the column exists in the table. Compare this to Hive-style explicit partitioning where the analyst MUST include the partition column in the WHERE clause to get a partition prune. Which one is friendlier to a downstream BI tool that auto-generates SQL? Which one is friendlier to a data engineer who wants to evolve the partition layout six months from now without rewriting consumer queries?

3. The lab's cutover plan keeps the seed external table around for 30 days post-cutover and archives the CSV prefix to Glacier rather than deleting it. Walk through the trade-off: storage cost vs. rollback insurance. At what volume of legacy data does the storage cost stop being trivial? What is the lightest-weight artifact you could keep that still gives you a rollback path?

## Exam-Style Practice

**Q1.** A data engineer migrated a daily CSV pipeline to Iceberg with hidden partitioning by `day(_ingestion_date)`. The legacy CSV stored `created_at` as epoch-seconds; the Iceberg side stores it as a Parquet TIMESTAMP preserving millisecond precision from the source system. The parity diff query uses `a.created_at = b.created_at` as the equality predicate and returns 1,200 mismatching rows. What is the right fix?

A) Reload the legacy CSV with millisecond timestamps so both sides have identical precision.

B) Reject the migration: the precision mismatch is a data quality issue that must be resolved at source.

C) Add a one-second tolerance to the predicate (`ABS(date_diff('second', a.created_at, b.created_at)) <= 1`) and re-check.

D) Accept the 1,200 mismatches as expected legacy precision drift and document them in the runbook.

<details><summary>Show answer</summary>

**C**.

- A) Wrong because the legacy data already exists; reloading it would require re-running three years of historical pipeline output, which is impractical and provides no real benefit over a tolerance-based diff.
- B) Wrong because the precision difference is intrinsic to how the legacy pipeline serialized timestamps. Rejecting the migration would block every legacy-to-modern migration that ever happens; tolerance windows are the canonical handling.
- C) Right because the tolerance window absorbs the precision drift without losing the ability to detect real drift (anything > 1 second still flags as a mismatch). This is the pattern Task 3 of the lab implements and is the canonical handling for cross-precision parity diffs.
- D) Wrong because accepting mismatches without analysis means the diff stops being a signal. A future genuine bug that produces sub-second drift will hide in the 1,200 known-not-bugs and the team will not notice.

</details>

**Q2.** A migration team is choosing between Iceberg hidden partitioning (`day(_ingestion_date)`) and Hive-style explicit partitioning (`PARTITIONED BY (ingestion_date)`). Their primary consumers are BI tools that auto-generate SQL based on a column-list manifest. Which is the right choice and why?

A) Hive explicit partitioning, because BI tools need the partition column in the schema to generate efficient queries.

B) Iceberg hidden partitioning, because the BI tool does not need to know about the partition column to get a partition prune.

C) Either works equivalently; the choice is style, not function.

D) Iceberg hidden partitioning, but only if the BI tool supports the Iceberg SDK; otherwise Hive explicit.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because Hive explicit partitioning requires the consumer to KNOW about the partition column and include it in WHERE; an auto-generated query that does not know about `ingestion_date` will scan the full table. The BI tool would need to be reconfigured every time the partition layout changes.
- B) Right because Iceberg hidden partitioning prunes partitions based on predicates on the source column (`created_at` in this lab's case), not on the partition column itself. The BI tool generates a query against the analyst-facing columns and Iceberg figures out which partitions to scan. The partition layout can evolve without breaking the BI manifest.
- C) Wrong because the two have materially different operational characteristics. Hive partitioning couples the query shape to the partition layout; Iceberg decouples them.
- D) Wrong because Iceberg is queried via the Athena (or Trino, or Spark) SQL layer; the BI tool does not need an Iceberg-specific SDK, it just needs to talk to the SQL engine. Iceberg hidden partitioning is the canonical choice for BI-driven consumers regardless of the SDK story.

</details>

**Q3.** A parity diff between a legacy CSV pipeline and a new Iceberg pipeline returns zero rows on a five-day overlap window. The team is debating whether this is sufficient evidence to decommission the legacy pipeline. Which statement most accurately captures the right move?

A) Zero rows is sufficient evidence; proceed with decommission immediately.

B) Zero rows is necessary but not sufficient; the team should also confirm downstream consumers and validate at least one nightly cycle post-cutover before deleting the legacy artifacts.

C) Zero rows is insufficient; a parity diff with no tolerance window is the only valid signal.

D) Zero rows is insufficient; the team should also run a 30-day comparison before any decommission.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because zero rows confirms the data matches but says nothing about downstream consumer readiness. A consumer pinned to the legacy table by name will break the moment the legacy view is dropped, regardless of parity.
- B) Right because a clean parity diff is the data-quality gate, not the operational gate. The cutover plan in Task 4 covers the additional operational steps: confirm downstream consumers (the Decommission Steps section), keep the legacy view available in a paused state for the rollback window (Rollback Plan), and validate at least one post-cutover cycle (Success Criteria). This is the canonical staged-cutover pattern.
- C) Wrong because tolerance-window diffs are correct and necessary when the two sides have different precision (epoch-seconds vs. milliseconds, as in this lab). A no-tolerance diff would produce false positives on every row.
- D) Wrong because a 30-day window is arbitrary; the right window is "long enough to cover the slowest downstream consumer's cycle plus one cycle of safety margin." For most analytics consumers that is the 5-day window the lab specifies. A 30-day requirement is overhead, not safety.

</details>
